# Day 1 — Data preparation and Logistic Regression baseline

**Project:** RetailMind AI — Customer Churn Prediction

**Goal:** Prepare the Telco Customer Churn data, inspect its quality and target balance, split it without leakage, build preprocessing, and establish a Logistic Regression baseline.

This notebook follows the project's original implementation. Run the cells from top to bottom; earlier-day steps are included where needed so this notebook can run independently.

## 0. Install dependencies

Install the libraries only if they are not already available in your environment. The line is intentionally commented so that running the notebook does not reinstall packages unnecessarily. If an import later reports a missing package, uncomment the command and run this cell once.

In [ ]:
# Run this cell if any required library is missing:
# %pip install pandas numpy scikit-learn joblib matplotlib xgboost tensorflow


## 1. Imports and project paths

Import the libraries and scikit-learn components used throughout the project. The code also defines the project paths and shared settings, so later cells can load the CSV, construct preprocessing/model pipelines, evaluate models and save trained artifacts consistently.

In [1]:
from pathlib import Path
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, roc_auc_score,
    classification_report, confusion_matrix
)

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
DATA_PATH = PROJECT_ROOT / "data" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR.mkdir(exist_ok=True)
REPORTS_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATA_PATH)
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. Put the Kaggle CSV inside the project's data/ folder."
    )


Project root: C:\Users\KIIT\OneDrive\Desktop\RetailMind
Dataset path: C:\Users\KIIT\OneDrive\Desktop\RetailMind\data\WA_Fn-UseC_-Telco-Customer-Churn.csv


## 2. Load and inspect the dataset — Day 1

Read the Telco Customer Churn CSV into a DataFrame and inspect its first rows, schema, descriptive statistics and missing-value counts. This first inspection helps confirm that the expected customer attributes and churn target are present, and highlights data-quality issues that must be handled before model training.

In [2]:
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())
display(df.info())
display(df.describe(include="all").T)
display(df.isna().sum().sort_values(ascending=False).head(15))
print("Target distribution:")
display(df["Churn"].value_counts())
display(df["Churn"].value_counts(normalize=True).rename("proportion"))


Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

None

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
customerID,7043,7043,7590-VHVEG,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gender,7043,2,Male,3555,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SeniorCitizen,7043.0,NaN,NaN,NaN,0.162147,0.368612,0.0,0.0,0.0,0.0,1.0
Partner,7043,2,No,3641,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Dependents,7043,2,No,4933,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tenure,7043.0,NaN,NaN,NaN,32.371149,24.559481,0.0,9.0,29.0,55.0,72.0
PhoneService,7043,2,Yes,6361,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MultipleLines,7043,3,No,3390,NaN,NaN,NaN,NaN,NaN,NaN,NaN
InternetService,7043,3,Fiber optic,3096,NaN,NaN,NaN,NaN,NaN,NaN,NaN
OnlineSecurity,7043,3,No,3498,NaN,NaN,NaN,NaN,NaN,NaN,NaN


customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
dtype: int64

Target distribution:


Churn
No     5174
Yes    1869
Name: count, dtype: int64

Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64

## 3. Clean data and create train/test split — Day 1

Convert `TotalCharges` into a numeric column, separate the predictors from the target, encode the churn label as 0/1, and create training and test partitions. The split is stratified so the proportion of customers who churn remains similar in both partitions. The test set is held back for final evaluation rather than model fitting.

In [3]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# customerID is an identifier, so it is excluded from model features.
X = df.drop(columns=["Churn", "customerID"]).copy()
y = df["Churn"].map({"No": 0, "Yes": 1}).astype("int32")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train churn rate:", round(y_train.mean(), 4))
print("Test churn rate:", round(y_test.mean(), 4))


Train shape: (5634, 19)
Test shape: (1409, 19)
Train churn rate: 0.2654
Test churn rate: 0.2654


## 4. Build reusable preprocessing pipeline — Day 1

Define the preprocessing steps used by the machine-learning models. Numeric columns are imputed and scaled, while categorical columns are imputed and one-hot encoded. Keeping these operations inside a scikit-learn pipeline ensures that transformations are fitted using training data and then applied consistently to validation and test data.

In [4]:
def make_preprocessor(X_reference):
    numeric_columns = X_reference.select_dtypes(include=["number"]).columns.tolist()
    categorical_columns = X_reference.select_dtypes(
        include=["object", "category", "bool"]
    ).columns.tolist()

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    return ColumnTransformer([
        ("num", numeric_pipeline, numeric_columns),
        ("cat", categorical_pipeline, categorical_columns)
    ])

preprocessor = make_preprocessor(X_train)


## 5. Day 1 — Logistic Regression baseline

Create and fit the first Logistic Regression pipeline as the Day 1 baseline. This gives the project a reference score that later models must improve on. Because churn is the minority class, the classifier uses class weighting to give churn examples greater influence during training.

In [5]:
baseline_model = Pipeline([
    ("preprocessor", make_preprocessor(X_train)),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced"))
])
baseline_model.fit(X_train, y_train)

def evaluate_classifier(model, X_eval, y_eval, model_name):
    predictions = model.predict(X_eval)
    probabilities = model.predict_proba(X_eval)[:, 1]
    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_eval, predictions),
        "Precision": precision_score(y_eval, predictions, zero_division=0),
        "Recall": recall_score(y_eval, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y_eval, probabilities)
    }

baseline_metrics = evaluate_classifier(
    baseline_model, X_test, y_test, "Logistic Regression Baseline"
)
display(pd.DataFrame([baseline_metrics]))
print(classification_report(y_test, baseline_model.predict(X_test), zero_division=0))

joblib.dump(baseline_model, MODELS_DIR / "churn_pipeline.pkl")
pd.DataFrame([baseline_metrics]).to_csv(REPORTS_DIR / "day1_baseline_metrics.csv", index=False)
print("Saved Day 1 baseline model and metrics.")


,Model,Accuracy,Precision,Recall,ROC-AUC
0,Logistic Regression Baseline,0.738112,0.504303,0.783422,0.841298


              precision    recall  f1-score   support

           0       0.90      0.72      0.80      1035
           1       0.50      0.78      0.61       374

    accuracy                           0.74      1409
   macro avg       0.70      0.75      0.71      1409
weighted avg       0.80      0.74      0.75      1409

Saved Day 1 baseline model and metrics.


## Day 1 — Completion

This notebook contains the executable project workflow through Day 1. Run all cells in order and review the printed metrics, tables and plots before recording results in the project README.